# Notebook 2 — Preprocessing

**Computer Vision CW1 — Diabetic Retinopathy Stage Detection**
Mohamed Shafran · NIBM (BSCCOMP24.2P)

This notebook turns every photo into the images the model will learn from. It does four things:

1. **Measures the camera shortcut** found in notebook 1 — how well the stage can be guessed from the camera alone — so the model has a baseline it must beat.
2. **Defines the preprocessing** in one file, `dr_preprocessing.py`. The Streamlit app will import the same file, so an uploaded photo is prepared exactly like the training images.
3. **Shows every step working**, with before/after figures and numbers.
4. **Builds three cached versions of the dataset** — A basic, B CLAHE + unsharp masking, C Ben Graham — so notebook 3 can compare them fairly (experiment E2).

**How to run:** set *Accelerator* to **None**. Attach two inputs: the **APTOS 2019** competition data and the **output of notebook 1** (*Add Input → Your Work → Notebooks → dr-01-explore-split*). Then **Run All**. It takes about 10 minutes.

**Leakage:** every preprocessing step uses only the pixels of the image it is working on. The two small models fitted here (sections 1 and 6) learn from the training split and are scored on the validation split. The test split is preprocessed like everything else, but no result is computed on it.

In [ ]:
# ============================================================
# Setup: imports, fixed seed, folders, the shared figure style
# ============================================================
import os, json, time, zipfile, importlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, recall_score

SEED = 42
np.random.seed(SEED)

OUT_DIR = Path("/kaggle/working")          # everything saved here becomes the notebook's output
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUT_DIR)                          # dr_preprocessing.py is written and imported here

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
STAGE_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]   # same as notebook 1
BLUE = "#2a78d6"
INK, INK_SOFT, MUTED, GRID, FLAG = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#d03b3b"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK_SOFT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "font.size": 10, "figure.dpi": 110, "savefig.bbox": "tight",
})

def tidy(ax):
    """Remove chart junk: drop the top/right frame, put a hairline grid behind the data."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    return ax

def picture(ax, img, title=None):
    """Show an image without axes."""
    ax.imshow(img)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color=INK_SOFT)

def find_file(name, root="/kaggle/input"):
    """Search the attached inputs for a file, wherever Kaggle mounted it."""
    for path, dirs, files in os.walk(root):
        if name in files:
            return Path(path) / name
        dirs[:] = [d for d in dirs if not d.lower().endswith("_images")]   # skip image folders
    raise FileNotFoundError(f"{name} not found under {root} - is that input attached?")

DATA = find_file("train.csv").parent
TRAIN_IMG_DIR = DATA / "train_images"
splits = pd.read_csv(find_file("splits.csv"))          # made by notebook 1

print("APTOS data :", DATA)
print("splits.csv :", find_file("splits.csv"))
print("images     :", len(splits), splits["split"].value_counts().to_dict())
missing = [c for c in splits["id_code"] if not (TRAIN_IMG_DIR / f"{c}.png").exists()]
assert not missing, f"{len(missing)} images listed in splits.csv are missing from disk"

## 1. The camera shortcut, measured

Notebook 1 (figure 11) found something important: the share of healthy (stage 0) eyes changes enormously with the image size — from almost none for some sizes to all of them for others. The APTOS images come from several clinics and cameras, and each camera produces its own image size, so the size tells us, roughly, which camera and clinic a photo came from. Some of them sent mostly healthy eyes, others mostly diseased ones.

That is a trap for a CNN. It can learn to recognise the camera — its colours, its sharpness, the shape of its field of view — instead of the lesions, and still score well here, because the same cameras appear in the test set. This is called **shortcut learning** (additional knowledge).

To see how deep the trap is, the simplest possible "model" is built below. For each image size it predicts the stage that was most common for that size in the **training** split. It never looks at the retina. Its scores on the **validation** split are the bar the CNN must clear — above all for the yes/no DR question.

The metrics used for every model in this project:

- **Accuracy** — the share of images given the right stage.
- **Macro-F1** — the F1-score of each stage, averaged so that every stage counts equally. A model that ignores the rare stages scores low even when its accuracy looks fine.
- **QWK** (quadratic weighted kappa, additional knowledge) — agreement on *ordered* classes: a 0→4 mistake is punished far more than a 1→2 mistake. 0 means no better than chance, 1 means perfect.
- For **DR yes/no**: accuracy, **sensitivity** (the share of DR eyes that are caught) and **specificity** (the share of healthy eyes correctly cleared).

In [ ]:
# The image size stands in for the camera. The six most common sizes cover most images;
# the rest are grouped as "other".
splits["camera"] = splits["width"].astype(str) + "x" + splits["height"].astype(str)
top_cameras = splits["camera"].value_counts().head(6).index.tolist()
splits["camera_group"] = splits["camera"].where(splits["camera"].isin(top_cameras), "other")

print("images per camera and split (every camera should appear in every split):")
print(pd.crosstab(splits["camera_group"], splits["split"])
        .reindex(index=top_cameras + ["other"], columns=["train", "val", "test"])
        .fillna(0).astype(int).to_string())


def scores(true, pred):
    """The metrics used for every model in this project: 5 stages, then DR yes/no."""
    true, pred = np.asarray(true), np.asarray(pred)
    dr_true, dr_pred = true > 0, pred > 0
    return {
        "accuracy": accuracy_score(true, pred),
        "macro_f1": f1_score(true, pred, labels=list(range(5)), average="macro", zero_division=0),
        "qwk": cohen_kappa_score(true, pred, weights="quadratic"),
        "dr_accuracy": accuracy_score(dr_true, dr_pred),
        "dr_sensitivity": recall_score(dr_true, dr_pred, zero_division=0),
        "dr_specificity": recall_score(~dr_true, ~dr_pred, zero_division=0),
    }


train = splits[splits["split"] == "train"]
val = splits[splits["split"] == "val"]

# "camera only": for each image size, the stage that was most common in the TRAINING split
stage_for_camera = train.groupby("camera")["diagnosis"].agg(lambda s: s.value_counts().idxmax())
most_common_stage = int(train["diagnosis"].value_counts().idxmax())
camera_guess = val["camera"].map(stage_for_camera).fillna(most_common_stage).astype(int)

baselines = pd.DataFrame(
    [scores(val["diagnosis"], np.zeros(len(val), dtype=int)),
     scores(val["diagnosis"], camera_guess)],
    index=["always 'No DR'", "camera only (image size)"])
baselines.index.name = "model"
baselines.to_csv(OUT_DIR / "baselines_val.csv")

print("\nvalidation scores of two models that never look at the retina:")
print(baselines.round(3).to_string())

In [ ]:
# Figure 12 - the shortcut in one picture
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.35, 1]})

# left: which stages each camera photographed
mix = (pd.crosstab(splits["camera"], splits["diagnosis"], normalize="index")
       .reindex(index=top_cameras, columns=range(5), fill_value=0) * 100)
camera_counts = splits["camera"].value_counts()
y, left = np.arange(len(mix)), np.zeros(len(mix))
for s in range(5):
    widths = mix[s].to_numpy()
    axes[0].barh(y, widths, left=left, height=0.62, color=STAGE_COLORS[s],
                 edgecolor="#fcfcfb", linewidth=0.8, label=f"{s} {CLASS_NAMES[s]}")
    if s == 0:                                         # label the healthy share, the one that swings
        for yi, w in zip(y, widths):
            if w >= 8:
                axes[0].text(w / 2, yi, f"{w:.0f}%", ha="center", va="center", fontsize=9, color=INK)
    left += widths
axes[0].set_yticks(y)
axes[0].set_yticklabels([f"{c}  (n={camera_counts[c]})" for c in top_cameras])
axes[0].invert_yaxis()
axes[0].set_xlim(0, 100)
axes[0].set_xlabel("Percent of that camera's images")
axes[0].set_title(f"Stage 0 is {mix[0].min():.0f}% to {mix[0].max():.0f}% of a camera's images")
axes[0].legend(frameon=False, ncols=5, fontsize=8.5, labelcolor=INK_SOFT,
               loc="upper center", bbox_to_anchor=(0.5, -0.15))
for side in ("top", "right"):
    axes[0].spines[side].set_visible(False)

# right: what the camera alone achieves on the validation split
metrics = ["accuracy", "macro_f1", "qwk", "dr_accuracy"]
x, width = np.arange(len(metrics)), 0.36
for k, (name, colour) in enumerate(zip(baselines.index, [MUTED, BLUE])):
    values = baselines.loc[name, metrics].to_numpy(dtype=float)
    bars = axes[1].bar(x + (k - 0.5) * width, values, width=width - 0.04, color=colour, label=name)
    for bar, v in zip(bars, values):
        axes[1].text(bar.get_x() + bar.get_width() / 2, max(v, 0) + 0.02, f"{v:.2f}",
                     ha="center", fontsize=9, color=INK_SOFT)
axes[1].set_xticks(x)
axes[1].set_xticklabels(["Accuracy", "Macro-F1", "QWK", "DR yes/no\naccuracy"])
lowest = float(baselines[metrics].to_numpy().min())
axes[1].set_ylim(min(0.0, lowest - 0.05), 1.12)
axes[1].set_ylabel("Score on the validation split")
dr_acc = baselines.loc["camera only (image size)", "dr_accuracy"]
axes[1].set_title(f"The camera alone gets DR yes/no {dr_acc:.0%} right")
axes[1].legend(frameon=False, labelcolor=INK_SOFT, loc="upper left")
tidy(axes[1])

plt.tight_layout()
plt.savefig(FIG_DIR / "fig12_camera_shortcut.png")
plt.show()

## 2. The preprocessing pipeline

Each step, and the reason for it.

**Step 1 — find the retina and give every photo the same window** *(retina segmentation, additional knowledge)*. A grey-level threshold separates the retina from the black border. The retina's *width* gives its radius: cameras that cut the retina off usually do so at the top and bottom, so the width is still the full diameter.

Some cameras show the whole round retina and others cut its top and bottom off flat. Because the camera is a shortcut (section 1), that shape alone could tell a CNN which camera took the photo. So every photo gets **the same window**: the full width of the retina, with the top and bottom trimmed flat at the level where the most cut-off cameras end (0.74 × the radius from the centre). That level is a property of the camera hardware, not of any label: it was measured on sample images and is checked against every image in section 4. Full-circle photos lose a strip at the top and bottom. The cut-off photos keep almost everything — and in the sample images, nearly all the diseased eyes were cut-off photos. A simpler alternative — the largest circle that fits inside every retina — was tried on sample images first and rejected, because on the cut-off photos it sliced through the optic disc.

**Step 2 — resize to 512 × 512.** Every photo gets the same size. Shrinking uses *area interpolation*, which averages each block of original pixels into one output pixel: a retina 2,000 px across becomes 512 px, so about 15 pixels are averaged into each one. This averaging is also the pipeline's **noise removal** — random sensor noise cancels out. No median or bilateral filter is added on purpose: microaneurysms, the earliest sign of DR, are only a few pixels wide at this size, and those filters are built to erase exactly such small dots.

**Step 3 — mask at 95 % of the radius.** Everything outside the window is set to black. Trimming the outer 5 % removes the dark rim at the edge of the lens. The margin applies to the flat top and bottom too, so they end up at 0.70 × the radius — safely inside even the most cut-off photos.

→ **Variant A (basic)** = steps 1–3. The colours are untouched.

**CLAHE — contrast adjustment** *(additional knowledge)*. *Histogram equalisation* spreads an image's grey levels over the full 0–255 range, so a dark, flat image gains contrast. Doing that to the whole image at once over-brightens some regions, so **CLAHE** (*Contrast Limited Adaptive Histogram Equalisation*) does it separately in each tile of an 8 × 8 grid and blends the tiles smoothly. *Contrast limited* means each tile's boost is capped (clip limit 2.0), so noise is not blown up. It is applied only to the **lightness** channel of the LAB colour space, so colours — and with them the colour of lesions — stay natural.

**Unsharp masking — edge enhancement** *(additional knowledge)*. A blurred copy of an image contains everything except the fine detail, so *image − blurred image* **is** the fine detail: vessel edges, microaneurysms, the borders of exudates. Adding it back makes those edges crisper:
`sharp = image + 1.0 × (image − blur(image, σ = 2 px))`. σ sets how fine the sharpened detail is; 2 px at 512 px matches small lesions and thin vessels.

→ **Variant B** = A + CLAHE + unsharp masking — the brief's "contrast adjustment and edge enhancement".

**Ben Graham's method — local colour normalisation.** It comes from the winner of the 2015 Kaggle diabetic retinopathy competition (it is defined in the lecturer's notebook, but never used there):
`out = 4 × (image − local average colour) + 128`, where the local average is a heavy blur (σ = radius ÷ 30 ≈ 8.5 px). Subtracting it removes slow changes — uneven lighting, dark edges, each camera's colour cast — and keeps the local detail, centred on mid-grey. Because it removes the colour cast, it may also weaken the camera shortcut. Section 6 measures whether it does.

→ **Variant C** = A + Ben Graham.

**One detail that matters — the masked blur** *(additional knowledge)*. Both the unsharp mask and Ben Graham's method subtract a blur. An ordinary blur near the edge of the window mixes in the black background, which leaves a false bright ring along the edge. The pipeline blurs only retina pixels: it divides the blurred image by a blurred copy of the mask (a *normalised convolution*), so the edge stays clean.

The cell below writes all of this to `dr_preprocessing.py`. Every function in it works on one image, using only that image's pixels.

In [ ]:
%%writefile dr_preprocessing.py
"""
dr_preprocessing.py - the ONE preprocessing file of this project.

It is used in two places, so every image is prepared in exactly the same way:
  * Kaggle notebook 2 (dr-02-preprocess), to build the cached training images;
  * the Streamlit prototype (app/), to prepare a photo a user uploads.

Every function works on a single image and uses only that image's own pixels.
Nothing is learned from other images, so nothing can leak between the
train, validation and test sets.

Three variants are produced from the same standardised image:
  A  basic          - retina centred, resized, same window shape for every photo
  B  CLAHE + unsharp - A + contrast adjustment + edge enhancement
  C  Ben Graham      - A + local colour normalisation (Graham, 2015)

Computer Vision CW1 - Diabetic Retinopathy Stage Detection - Mohamed Shafran
"""
import cv2
import numpy as np

# ---------------------------------------------------------------------------
# Settings (each one is explained in notebook 2)
# ---------------------------------------------------------------------------
SIZE = 512              # every output image is SIZE x SIZE pixels
MASK_SCALE = 0.95       # keep 95% of the radius: drops the dark rim at the edge of the lens
FOV_BAND = 0.74         # keep only rows within 0.74 x radius of the centre - the level at which
                        # the most cut-off cameras end - so every photo gets the same shape
BG_TOL = 10             # grey level at or below which a pixel counts as black background
ROW_FRACTION = 0.02     # a row or column is part of the retina if >2% of its pixels are bright

CLAHE_CLIP = 2.0        # CLAHE contrast limit: higher = stronger boost, but more noise
CLAHE_TILES = (8, 8)    # CLAHE equalises each tile of an 8 x 8 grid separately
UNSHARP_SIGMA = 2.0     # size (in pixels) of the fine detail the unsharp mask sharpens
UNSHARP_AMOUNT = 1.0    # how much of that detail is added back (1.0 = detail doubled)
BEN_GRAHAM_SIGMA = SIZE / 2 / 30                 # radius / 30, as in Graham (2015): ~8.5 px

VARIANTS = {"A": "basic", "B": "CLAHE + unsharp mask", "C": "Ben Graham"}
PNG_SETTINGS = [cv2.IMWRITE_PNG_COMPRESSION, 3]   # lossless; 3 = good balance of size and speed


def window_mask(size=SIZE, scale=MASK_SCALE, band=FOV_BAND):
    """The field of view every image is given: a centred disc with its top and bottom
    cut off flat. `scale` shrinks the disc and the band together."""
    centre = (size - 1) / 2
    yy, xx = np.ogrid[:size, :size]
    disc = (xx - centre) ** 2 + (yy - centre) ** 2 <= (scale * size / 2) ** 2
    rows = np.abs(yy - centre) <= scale * band * size / 2
    return disc & rows


MASK = window_mask()                            # the same window for every image
INNER = window_mask(SIZE, MASK_SCALE * 0.9)     # a smaller window, away from the mask edge


# ---------------------------------------------------------------------------
# Reading
# ---------------------------------------------------------------------------
def load_rgb(source):
    """Read an image from a file path or from raw bytes (an upload) and return RGB uint8."""
    if isinstance(source, (bytes, bytearray)):
        data = np.frombuffer(source, np.uint8)
    else:
        data = np.fromfile(str(source), np.uint8)
    bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)       # also drops an alpha channel
    if bgr is None:
        raise ValueError("the file could not be read as an image")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


# ---------------------------------------------------------------------------
# Steps 1-3: the same field of view for every photo  ->  variant A
# ---------------------------------------------------------------------------
def retina_box(rgb, tol=BG_TOL):
    """Edges (top, bottom, left, right) of the retina and how many pixels it covers.
    A row or column counts only if more than ROW_FRACTION of it is bright, so a few
    stray bright pixels in the black border cannot stretch the box."""
    bright = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY) > tol
    rows = np.where(bright.mean(axis=1) > ROW_FRACTION)[0]
    cols = np.where(bright.mean(axis=0) > ROW_FRACTION)[0]
    if len(rows) == 0 or len(cols) == 0:
        return None, 0                                # nothing bright in the whole image
    return (int(rows[0]), int(rows[-1]), int(cols[0]), int(cols[-1])), int(bright.sum())


def find_retina(rgb):
    """Centre and radius of the retina, plus its height/width ratio.
    The radius is half the retina's WIDTH: cameras that cut the retina off do so at the
    top and bottom, so the width is the full diameter. The ratio says how much was cut
    (1.0 = a complete circle)."""
    box, retina_pixels = retina_box(rgb)
    height, width = rgb.shape[:2]
    if box is None:                                   # fall back to the whole frame
        return (width - 1) / 2, (height - 1) / 2, min(height, width) / 2, height * width, 1.0
    top, bottom, left, right = box
    box_height, box_width = bottom - top + 1, right - left + 1
    return ((left + right) / 2, (top + bottom) / 2, box_width / 2, retina_pixels,
            box_height / box_width)


def standardise(rgb):
    """Variant A: a square around the retina, resized to SIZE x SIZE, with everything
    outside the common window blacked out. Returns the image and a dict describing it."""
    cx, cy, radius, retina_pixels, box_ratio = find_retina(rgb)
    side = max(int(round(2 * radius)), 1)
    x0, y0 = int(round(cx - radius)), int(round(cy - radius))

    height, width = rgb.shape[:2]
    pad = max(0, -x0, -y0, x0 + side - width, y0 + side - height)
    if pad:                                           # the square reaches past the photo's edge
        rgb = cv2.copyMakeBorder(rgb, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
        x0, y0 = x0 + pad, y0 + pad
    square = rgb[y0:y0 + side, x0:x0 + side]

    # INTER_AREA averages each block of pixels when shrinking (this also removes sensor
    # noise); INTER_CUBIC is the better choice for the rare photo smaller than SIZE.
    shrinking = side >= SIZE
    img = cv2.resize(square, (SIZE, SIZE),
                     interpolation=cv2.INTER_AREA if shrinking else cv2.INTER_CUBIC)
    img[~MASK] = 0

    window_area = MASK.sum() * (radius / (SIZE / 2)) ** 2      # in original pixels
    return img, {"radius_px": float(radius), "box_ratio": float(box_ratio),
                 "fov_kept": float(min(window_area / max(retina_pixels, 1), 1.0)),
                 "upscaled": not shrinking}


# ---------------------------------------------------------------------------
# Enhancement
# ---------------------------------------------------------------------------
def masked_blur(img, sigma, mask=MASK):
    """Gaussian blur that averages ONLY retina pixels (a 'normalised convolution').
    An ordinary blur would mix the black background into the edge of the retina and
    create a false bright ring there once the blur is subtracted."""
    weight = mask.astype(np.float32)
    total = cv2.GaussianBlur(img.astype(np.float32) * weight[..., None], (0, 0), sigma)
    norm = cv2.GaussianBlur(weight, (0, 0), sigma)[..., None]
    return total / np.maximum(norm, 1e-6)


def clahe_unsharp(img_a, return_steps=False):
    """Variant B: contrast adjustment (CLAHE) followed by edge enhancement (unsharp mask)."""
    # 1. CLAHE on the lightness channel of LAB only, so the colours stay natural
    lightness, a, b = cv2.split(cv2.cvtColor(img_a, cv2.COLOR_RGB2LAB))
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILES)
    contrast = cv2.cvtColor(cv2.merge([clahe.apply(lightness), a, b]), cv2.COLOR_LAB2RGB)
    contrast[~MASK] = 0

    # 2. unsharp mask: (image - blurred image) is the fine detail; add it back once more
    detail = contrast.astype(np.float32) - masked_blur(contrast, UNSHARP_SIGMA)
    sharp = np.clip(contrast + UNSHARP_AMOUNT * detail, 0, 255).astype(np.uint8)
    sharp[~MASK] = 0
    return (contrast, sharp) if return_steps else sharp


def ben_graham(img_a):
    """Variant C: subtract the local average colour, multiply by 4, centre on grey 128.
    Slow changes (uneven lighting, each camera's colour cast) disappear;
    local detail (vessels, lesions) stays."""
    local_average = masked_blur(img_a, BEN_GRAHAM_SIGMA)
    out = np.clip(4.0 * (img_a.astype(np.float32) - local_average) + 128.0, 0, 255).astype(np.uint8)
    out[~MASK] = 128                                  # neutral grey outside, as in the original
    return out


# ---------------------------------------------------------------------------
# One call for the app, one for building the cache
# ---------------------------------------------------------------------------
def preprocess(rgb, variant="A"):
    """The full pipeline for one image and one variant ('A', 'B' or 'C')."""
    if variant not in VARIANTS:
        raise ValueError(f"unknown variant {variant!r}; use one of {list(VARIANTS)}")
    img_a, _ = standardise(rgb)
    if variant == "B":
        return clahe_unsharp(img_a)
    if variant == "C":
        return ben_graham(img_a)
    return img_a


def all_variants(rgb):
    """All three variants from a single read of the photo (used to build the cache)."""
    img_a, info = standardise(rgb)
    return {"A": img_a, "B": clahe_unsharp(img_a), "C": ben_graham(img_a)}, info


def image_stats(img):
    """Numbers describing one processed image, measured inside the circle only."""
    pixels = img[MASK].astype(np.float32)
    grey = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    laplacian = cv2.Laplacian(grey, cv2.CV_32F)
    means, spreads = pixels.mean(axis=0), pixels.std(axis=0)
    return {"mean_r": float(means[0]), "mean_g": float(means[1]), "mean_b": float(means[2]),
            "std_r": float(spreads[0]), "std_g": float(spreads[1]), "std_b": float(spreads[2]),
            "contrast": float(grey[MASK].std()),
            "sharpness": float(laplacian[INNER].var())}


In [ ]:
import dr_preprocessing as prep
importlib.reload(prep)            # picks up changes if the cell above is edited and re-run

# a quick self-test on one real image before processing thousands
first = splits.iloc[0]
rgb = prep.load_rgb(TRAIN_IMG_DIR / f"{first.id_code}.png")
start = time.time()
variants, info = prep.all_variants(rgb)
print(f"one {rgb.shape[1]}x{rgb.shape[0]} photo processed in {1000 * (time.time() - start):.0f} ms")
print("what happened to it:", {k: round(v, 3) if isinstance(v, float) else v for k, v in info.items()})

for name, img in variants.items():
    assert img.shape == (prep.SIZE, prep.SIZE, 3) and img.dtype == np.uint8, name
assert (variants["A"][~prep.MASK] == 0).all() and (variants["C"][~prep.MASK] == 128).all()
assert not np.array_equal(variants["A"], variants["B"]) and not np.array_equal(variants["A"], variants["C"])
for name in "ABC":            # the app calls preprocess(); it must give exactly the cached result
    assert np.array_equal(prep.preprocess(rgb, name), variants[name]), name
print("self-test passed: three 512x512 variants, masks correct, app path identical")

## 3. See it working

Four figures: one photo through every step, the three variants for one photo of each stage, a close-up of lesions, and what CLAHE does to the darkest training photo. Only training images are used for figures.

In figure 16, the *lightness spread* is the interquartile range (IQR): the width of the middle half of the pixel values. A wider spread means more contrast.

In [ ]:
# Figure 13 - one photo through every step
def display_copy(rgb, width=640):
    """A smaller copy of a large photo, only for showing it in a figure."""
    scale = width / rgb.shape[1]
    small = cv2.resize(rgb, (width, max(1, int(round(rgb.shape[0] * scale)))),
                       interpolation=cv2.INTER_AREA)
    return small, scale

# prefer a stage-2 training photo from a camera that cuts the retina off (height/width < 0.9)
candidates = train[train["diagnosis"] == 2].sample(frac=1, random_state=SEED).head(40)
example, example_rgb = None, None
for row in candidates.itertuples():
    rgb = prep.load_rgb(TRAIN_IMG_DIR / f"{row.id_code}.png")
    if prep.find_retina(rgb)[4] < 0.9:
        example, example_rgb = row, rgb
        break
if example is None:                                   # no cut-off photo: take the first one
    example = next(candidates.itertuples())
    example_rgb = prep.load_rgb(TRAIN_IMG_DIR / f"{example.id_code}.png")

img_a, info = prep.standardise(example_rgb)
after_clahe, img_b = prep.clahe_unsharp(img_a, return_steps=True)
img_c = prep.ben_graham(img_a)

# outline of the common window, mapped back onto the original photo
cx, cy, radius, _, ratio = prep.find_retina(example_rgb)
contours, _ = cv2.findContours(prep.MASK.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
outline = contours[0][:, 0, :].astype(np.float32)
outline = (outline + 0.5) * (2 * radius / prep.SIZE) + np.array([cx - radius, cy - radius])
shown, scale = display_copy(example_rgb)

fig, axes = plt.subplots(1, 5, figsize=(18, 4.3))
picture(axes[0], shown, f"1. Original {example_rgb.shape[1]}x{example_rgb.shape[0]}\n(yellow: the window kept)")
axes[0].plot(np.append(outline[:, 0], outline[0, 0]) * scale,
             np.append(outline[:, 1], outline[0, 1]) * scale, color="#eda100", linewidth=1.8)
picture(axes[1], img_a, "2. Variant A\ncentred, 512 px, common window")
picture(axes[2], after_clahe, "3. CLAHE\n(contrast adjustment)")
picture(axes[3], img_b, "4. Variant B\n+ unsharp mask (edges)")
picture(axes[4], img_c, "5. Variant C\nBen Graham")
fig.suptitle(f"One stage-{example.diagnosis} training photo through the pipeline "
             f"(retina height/width {ratio:.2f})", fontsize=13, fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.savefig(FIG_DIR / "fig13_pipeline_steps.png")
plt.show()

In [ ]:
# Figure 14 - the three variants for one training photo of each stage
column_titles = ["Original", "A  basic", "B  CLAHE + unsharp", "C  Ben Graham"]
fig, axes = plt.subplots(5, 4, figsize=(11, 14))
for stage in range(5):
    pool = train[train["diagnosis"] == stage]
    if pool.empty:
        for ax in axes[stage]:
            ax.axis("off")
        continue
    row = pool.sample(1, random_state=SEED).iloc[0]
    rgb = prep.load_rgb(TRAIN_IMG_DIR / f"{row.id_code}.png")
    variants, _ = prep.all_variants(rgb)
    for col, img in enumerate([display_copy(rgb, 512)[0], variants["A"], variants["B"], variants["C"]]):
        ax = axes[stage, col]
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if stage == 0:
            ax.set_title(column_titles[col], fontsize=11, color=INK)
    axes[stage, 0].set_ylabel(f"{stage} - {CLASS_NAMES[stage]}", fontsize=10, color=INK)
fig.suptitle("The three variants, one training photo per stage", fontsize=13, fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.975))
plt.savefig(FIG_DIR / "fig14_variants_by_stage.png")
plt.show()

In [ ]:
# Figure 15 - a close-up, where lesions and vessel edges can actually be seen
PATCH = 128
row = train[train["diagnosis"] >= 3].sample(1, random_state=SEED).iloc[0]
variants, _ = prep.all_variants(prep.load_rgb(TRAIN_IMG_DIR / f"{row.id_code}.png"))

# find the patch with the most local detail in variant C ...
detail = cv2.GaussianBlur((variants["C"].astype(np.float32).mean(axis=2) - 128) ** 2, (0, 0), 16)
# ... away from the optic disc (the brightest blob), so it shows lesions rather than the disc
glow = cv2.GaussianBlur(cv2.cvtColor(variants["A"], cv2.COLOR_RGB2GRAY).astype(np.float32), (0, 0), 15)
disc_y, disc_x = np.unravel_index(np.argmax(glow), glow.shape)
yy, xx = np.mgrid[:prep.SIZE, :prep.SIZE]
centre = (prep.SIZE - 1) / 2
allowed = (((yy - centre) ** 2 + (xx - centre) ** 2 <= (0.55 * prep.SIZE / 2) ** 2)
           & ((yy - disc_y) ** 2 + (xx - disc_x) ** 2 > 130 ** 2))
if allowed.any():
    detail[~allowed] = -1
py, px = np.unravel_index(np.argmax(detail), detail.shape)
y0 = int(np.clip(py - PATCH // 2, 0, prep.SIZE - PATCH))
x0 = int(np.clip(px - PATCH // 2, 0, prep.SIZE - PATCH))

fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
picture(axes[0], variants["A"], "Where the close-up is taken")
axes[0].add_patch(Rectangle((x0, y0), PATCH, PATCH, fill=False, edgecolor="white", linewidth=1.6))
for ax, name, label in zip(axes[1:], "ABC", ["A  basic", "B  CLAHE + unsharp", "C  Ben Graham"]):
    ax.imshow(variants[name][y0:y0 + PATCH, x0:x0 + PATCH], interpolation="nearest")
    ax.axis("off")
    ax.set_title(label, fontsize=10, color=INK_SOFT)
fig.suptitle(f"Close-up of a stage-{int(row['diagnosis'])} eye: the same {PATCH}x{PATCH} px patch "
             "in each variant", fontsize=13, fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.savefig(FIG_DIR / "fig15_closeup.png")
plt.show()

In [ ]:
# Figure 16 - CLAHE on the darkest training photo (brightness measured in notebook 1);
# photos darker than 20 are skipped - they are almost black and show nothing
row = train[train["brightness"] >= 20].nsmallest(1, "brightness").iloc[0]
img_a, _ = prep.standardise(prep.load_rgb(TRAIN_IMG_DIR / f"{row.id_code}.png"))
after_clahe, _ = prep.clahe_unsharp(img_a, return_steps=True)

lightness_before = cv2.cvtColor(img_a, cv2.COLOR_RGB2LAB)[..., 0][prep.MASK]
lightness_after = cv2.cvtColor(after_clahe, cv2.COLOR_RGB2LAB)[..., 0][prep.MASK]
iqr_before = np.subtract(*np.percentile(lightness_before, [75, 25]))
iqr_after = np.subtract(*np.percentile(lightness_after, [75, 25]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1, 1, 1.6]})
picture(axes[0], img_a, "Before (variant A)")
picture(axes[1], after_clahe, "After CLAHE")
bins = np.arange(0, 257, 4)
axes[2].hist(lightness_before, bins=bins, histtype="step", linewidth=2, color=MUTED, label="before (A)")
axes[2].hist(lightness_after, bins=bins, histtype="step", linewidth=2, color=BLUE, label="after CLAHE")
axes[2].set_xlim(0, 255)
axes[2].set_xlabel("Lightness (L channel of LAB, 0-255)")
axes[2].set_ylabel("Pixels inside the window")
axes[2].set_title(f"Lightness spread (IQR): {iqr_before:.0f} -> {iqr_after:.0f} levels")
axes[2].legend(frameon=False, labelcolor=INK_SOFT)
tidy(axes[2])
fig.suptitle(f"CLAHE on the darkest training photo ({row['id_code']})", fontsize=13,
             fontweight="bold", color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.savefig(FIG_DIR / "fig16_clahe_histogram.png")
plt.show()

## 4. Build the cache

All images — train, validation and test — go through the same deterministic pipeline. Each variant is saved as lossless PNG files inside **one zip archive per variant**. Three archives are easier for Kaggle to store, and for notebook 3 to load, than 10,000 loose files.

The same loop records a few numbers for every processed image, used in sections 5 and 6 and later in the error analysis. It runs on all CPU cores at once, because OpenCV releases Python's interpreter lock while it works.

In [ ]:
# Process every image once, write the three variants into three zip archives
archives = {name: zipfile.ZipFile(OUT_DIR / f"preprocessed_{name}.zip", "w", zipfile.ZIP_STORED)
            for name in prep.VARIANTS}          # PNG is already compressed, so the zip just stores

def process(id_code):
    """Read one photo, make all three variants, encode them as PNG, measure them."""
    variants, info = prep.all_variants(prep.load_rgb(TRAIN_IMG_DIR / f"{id_code}.png"))
    encoded = {name: cv2.imencode(".png", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), prep.PNG_SETTINGS)[1].tobytes()
               for name, img in variants.items()}
    record = {"id_code": id_code, **info}
    for name, img in variants.items():
        record.update({f"{name}_{key}": value for key, value in prep.image_stats(img).items()})
    return id_code, encoded, record

records, start, CHUNK = [], time.time(), 250
ids = splits["id_code"].tolist()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    for i in range(0, len(ids), CHUNK):                  # in chunks, so memory stays small
        for id_code, encoded, record in pool.map(process, ids[i:i + CHUNK]):
            for name, data in encoded.items():
                archives[name].writestr(f"{id_code}.png", data)
            records.append(record)
        print(f"  {min(i + CHUNK, len(ids)):>5} / {len(ids)} images   {time.time() - start:5.0f} s")
for archive in archives.values():
    archive.close()

# read the archives back: every image must be there, and must decode to 512 x 512
for name in prep.VARIANTS:
    with zipfile.ZipFile(OUT_DIR / f"preprocessed_{name}.zip") as archive:
        names = archive.namelist()
        assert len(names) == len(ids), f"variant {name}: {len(names)} files for {len(ids)} images"
        check = cv2.imdecode(np.frombuffer(archive.read(names[0]), np.uint8), cv2.IMREAD_COLOR)
        assert check.shape == (prep.SIZE, prep.SIZE, 3)
    size_mb = (OUT_DIR / f"preprocessed_{name}.zip").stat().st_size / 1e6
    print(f"preprocessed_{name}.zip : {len(names)} images, {size_mb:,.0f} MB")

table = splits.merge(pd.DataFrame(records), on="id_code")
table.to_csv(OUT_DIR / "preprocess_stats.csv", index=False)
print(f"\nall {len(table)} images processed in {time.time() - start:.0f} s")

In [ ]:
# How much of each camera's visible retina does the common window keep?
fov = (table.groupby("camera_group")
       .agg(images=("id_code", "size"),
            height_to_width=("box_ratio", "median"),
            share_kept=("fov_kept", "median"))
       .reindex(top_cameras + ["other"]))
print("per camera (medians): retina height/width, and share of the visible retina kept")
print(fov.round(3).to_string())

band_edge = prep.MASK_SCALE * prep.FOV_BAND
cut_more = int((table["box_ratio"] < band_edge).sum())
print(f"\nphotos cut off more than the common window ({band_edge:.3f} x radius): "
      f"{cut_more} ({100 * cut_more / len(table):.1f}%) - these keep a thin black strip inside the window")
print("photos enlarged because their retina was smaller than 512 px:", int(table["upscaled"].sum()))

## 5. Did the enhancement do what it should?

Two numbers per image, measured inside the window:

- **Contrast** — the spread (standard deviation) of grey levels. Higher means more separation between dark and bright structures.
- **Sharpness** — the variance of the Laplacian, the same blur measure used in notebook 1. Higher means crisper edges.

B should raise both. C is different by design: it removes the slow brightness changes and keeps only local detail, so its contrast measures detail rather than overall brightness range.

In [ ]:
# Figure 17 - contrast and sharpness of every processed image, per variant
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
labels = ["A  basic", "B  CLAHE + unsharp", "C  Ben Graham"]
quality = {}
for ax, measure in zip(axes, ["contrast", "sharpness"]):
    data = [table[f"{name}_{measure}"].to_numpy() for name in prep.VARIANTS]
    parts = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.5)
    for box in parts["boxes"]:
        box.set(facecolor="#86b6ef", edgecolor=INK_SOFT)
    for key in ("whiskers", "caps"):
        for line in parts[key]:
            line.set(color=INK_SOFT)
    for line in parts["medians"]:
        line.set(color=INK, linewidth=2)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(labels)
    medians = [float(np.median(d)) for d in data]
    quality[measure] = dict(zip(prep.VARIANTS, medians))
    ax.set_title(f"{measure.capitalize()} - medians {medians[0]:.0f} / {medians[1]:.0f} / {medians[2]:.0f}")
    if measure == "sharpness":
        ax.set_yscale("log")
        ax.set_ylabel("Laplacian variance (log scale)")
    else:
        ax.set_ylabel("Std of grey levels")
    tidy(ax)
fig.suptitle(f"Measured on all {len(table)} images, inside the window", fontsize=12, color=INK_SOFT)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig17_quality_numbers.png")
plt.show()

print(pd.DataFrame(quality).round(1).to_string())

## 6. Does the preprocessing hide the camera?

A **random forest** (additional knowledge: many small decision trees that vote — a fast, standard classifier for tables of numbers) is given seven numbers per image: the average and spread of red, green and blue inside the window, and the sharpness. It has to say which of the six common cameras took the photo. It learns on the training split and is scored on the validation split, once per variant.

If the forest can still tell the cameras apart, a CNN certainly can. Variant A keeps the original colours, so it shows how recognisable the cameras are to begin with. One caution: the cameras also differ in their mix of stages (section 1), so part of what the forest detects may be disease rather than camera. The comparison *between* variants is still fair, because the images are the same in all three.

In [ ]:
# Figure 18 - how well a simple classifier recognises the camera, per variant
features = ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b", "sharpness"]
known = table[table["camera_group"] != "other"]
fit_on, score_on = known[known["split"] == "train"], known[known["split"] == "val"]
most_common_camera = fit_on["camera_group"].value_counts().idxmax()
chance = float((score_on["camera_group"] == most_common_camera).mean())

camera_accuracy = {}
for name in prep.VARIANTS:
    columns = [f"{name}_{f}" for f in features]
    forest = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    forest.fit(fit_on[columns], fit_on["camera_group"])
    camera_accuracy[name] = float(accuracy_score(score_on["camera_group"], forest.predict(score_on[columns])))
print("camera recognised on the validation split:",
      {k: f"{v:.1%}" for k, v in camera_accuracy.items()}, f"| always guessing {most_common_camera}: {chance:.1%}")

fig, ax = plt.subplots(figsize=(7.6, 4.4))
values = [camera_accuracy[name] for name in prep.VARIANTS]
bars = ax.bar(range(3), values, width=0.55, color=BLUE)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.0%}", ha="center", fontsize=10, color=INK_SOFT)
ax.axhline(chance, color=INK_SOFT, linestyle="--", linewidth=1)
fig.text(0.5, -0.03, f"Dashed line: always guessing the commonest camera ({chance:.0%}).",
         ha="center", fontsize=9, color=INK_SOFT)
ax.set_xticks(range(3))
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Camera recognised (validation accuracy)")
hardest = min(camera_accuracy, key=camera_accuracy.get)
if camera_accuracy[hardest] < camera_accuracy["A"]:
    ax.set_title(f"Variant {hardest} hides the camera best: {camera_accuracy[hardest]:.0%} "
                 f"vs {camera_accuracy['A']:.0%} for A")
else:
    ax.set_title(f"No variant hides the camera better than A ({camera_accuracy['A']:.0%})")
tidy(ax)
plt.savefig(FIG_DIR / "fig18_camera_signature.png")
plt.show()

## 7. Save the results

Everything notebook 3 needs is in this notebook's output, so notebook 3 attaches **only this notebook** as input:

- **`preprocessed_A.zip`, `preprocessed_B.zip`, `preprocessed_C.zip`** — the cached 512 × 512 images.
- **`splits.csv`** — notebook 1's split plus the camera columns (`camera`, `camera_group`), used for the per-camera evaluation later.
- **`preprocess_stats.csv`** — the numbers measured for every image and variant.
- **`baselines_val.csv`** — the two "no retina" baselines from section 1.
- **`dr_preprocessing.py`** — the preprocessing module. Download it into `app/` for the prototype.
- **`preprocessing_summary.json`** — the headline numbers for the report.

In [ ]:
splits.to_csv(OUT_DIR / "splits.csv", index=False)

summary = {
    "images_processed": int(len(table)),
    "settings": {"size": prep.SIZE, "mask_scale": prep.MASK_SCALE, "fov_band": prep.FOV_BAND,
                 "clahe_clip": prep.CLAHE_CLIP, "clahe_tiles": list(prep.CLAHE_TILES),
                 "unsharp_sigma": prep.UNSHARP_SIGMA, "unsharp_amount": prep.UNSHARP_AMOUNT,
                 "ben_graham_sigma": round(prep.BEN_GRAHAM_SIGMA, 2)},
    "window_share_of_square": round(float(prep.MASK.mean()), 3),
    "median_share_of_retina_kept": round(float(table["fov_kept"].median()), 3),
    "share_kept_by_camera": {k: round(float(v), 3) for k, v in fov["share_kept"].dropna().items()},
    "height_to_width_by_camera": {k: round(float(v), 3) for k, v in fov["height_to_width"].dropna().items()},
    "images_per_camera_and_split": {cam: {sp: int(n) for sp, n in row.items()} for cam, row in
                                    pd.crosstab(splits["camera_group"], splits["split"]).iterrows()},
    "stage0_share_by_camera": {c: round(float(mix.loc[c, 0]), 1) for c in top_cameras},
    "photos_cut_more_than_window": cut_more,
    "photos_upscaled": int(table["upscaled"].sum()),
    "baselines_val": {model: {k: round(float(v), 3) for k, v in row.items()}
                      for model, row in baselines.iterrows()},
    "median_contrast": {k: round(v, 1) for k, v in quality["contrast"].items()},
    "median_sharpness": {k: round(v, 1) for k, v in quality["sharpness"].items()},
    "camera_recognition_val": {k: round(v, 3) for k, v in camera_accuracy.items()},
    "camera_recognition_chance": round(chance, 3),
    "archive_mb": {name: round((OUT_DIR / f"preprocessed_{name}.zip").stat().st_size / 1e6)
                   for name in prep.VARIANTS},
    "seed": SEED,
}
with open(OUT_DIR / "preprocessing_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nfiles written:", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))

## 8. What to do next

1. **Save Version → Save & Run All**, so the outputs are stored and can be attached to notebook 3.
2. Send back `preprocessing_summary.json` and figures 12–18.
3. Download `dr_preprocessing.py` from the output into `CW1_DR_Stage_Detection/app/`, and this notebook into `kaggle_notebooks/`.

**For the report**, this notebook provides the camera-shortcut figure and baseline table, the step-by-step pipeline figure, the variants per stage, the close-up, the CLAHE histogram, the quality numbers, and the camera-recognition test.

Notebook 3 will train the models on these cached images and compare the three variants (experiment E2).